In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: bronze_sales
position:
  x: 768
  y: 365
description:
  text: Load all data from the bronze_sales table.
  hash: 481e1d8c
previewCodeHash: 67ec2431672d9cab
previewMode: "1000"
config:
  table_source:
    tableName: cyntexa_dev.medallion_day_6.bronze_sales
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "cyntexa_dev.medallion_day_6.bronze_sales"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_0.data"])

In [0]:
"""
id: unique_1
template: unique
templateVersion: 1.0.0
name: Distinct rows
position:
  x: 1028
  y: 365
previewCodeHash: ba84bc1a0463a282
previewMode: "1000"
config:
  unique_by_all_columns: true
  columns: []
  sort_expressions: []
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F
from pyspark.sql import Window

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError("Unique operator requires input 'data'")
    unique_by_all_columns = config.get("unique_by_all_columns", True)
    columns = config.get("columns", [])
    if not isinstance(columns, list):
        columns = []
    else:
        columns = [c for c in columns if isinstance(c, str)]
    sort_expressions = config.get("sort_expressions", [])

    if unique_by_all_columns:
        columns = df.columns
    elif not columns:
        return {"unique_data": df}

    order_idx = "__lb_orig_order__"
    while order_idx in df.columns:
        order_idx = order_idx + "_"
    df = df.withColumn(order_idx, F.monotonically_increasing_id())

    if unique_by_all_columns or not sort_expressions:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    order_cols = []
    for sort_def in sort_expressions:
        if not isinstance(sort_def, dict):
            continue
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        if not raw_expr:
            continue
        direction = sort_def.get("sortBy", "ASC")
        col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        if direction == "DESC":
            col = col.desc_nulls_last()
        elif direction == "ASC":
            col = col.asc_nulls_last()
        order_cols.append(col)
    
    if not order_cols:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    rn = "__lb_row_number__"
    while rn in df.columns:
        rn = rn + "_"

    window = Window.partitionBy(*[F.col(c) for c in columns]).orderBy(*order_cols, F.col(order_idx))
    tagged = df.withColumn(rn, F.row_number().over(window))
    unique = tagged.filter(F.col(rn) == 1).orderBy(order_idx).drop(rn, order_idx)
    return {"unique_data": unique}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "unique_by_all_columns": True,
    "columns": [],
    "sort_expressions": []
}
inputs = {
    "data": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["unique_1.unique_data"] = out["unique_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["unique_1.unique_data"])

In [0]:
"""
id: prepare_2
template: prepare
templateVersion: 1.0.0
name: date string ->Date
position:
  x: 1288
  y: 365
description:
  text: Convert the 'order_date' column to date type, setting errors to null.
  hash: 616a48c3
previewCodeHash: c69d9c35188ba81c
previewMode: "1000"
config:
  actions:
    - type: cast
      column: order_date
      to: date
      on_error: "null"
input:
  - node: unique_1
    input_port: data
    output_port: unique_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "cast",
            "column": "order_date",
            "to": "date",
            "on_error": "null"
        }
    ]
}
inputs = {
    "data": ctx["unique_1.unique_data"]
}
out = run(config, inputs, spark)
ctx["prepare_2.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["prepare_2.prepared_data"])

In [0]:
"""
id: filter_3
template: filter
templateVersion: 2.0.0
name: remove nul dates
position:
  x: 1548
  y: 365
description:
  text: remove null dates
  hash: c71bef48
previewCodeHash: e4417b9775c53cef
previewMode: "1000"
config:
  condition: order_date IS NOT NULL
input:
  - node: prepare_2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "order_date IS NOT NULL"
}
inputs = {
    "data": ctx["prepare_2.prepared_data"]
}
out = run(config, inputs, spark)
ctx["filter_3.filtered_data"] = out["filtered_data"]
ctx["filter_3.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["filter_3.filtered_data"])
    display(ctx["filter_3.excluded_data"])

In [0]:
"""
id: filter_5
template: filter
templateVersion: 2.0.0
name: Remove null customer/order ids
position:
  x: 1808
  y: 520
description:
  text: Keep rows where customer_id and order_id are not missing; separate others.
  hash: 066f4d40
previewCodeHash: 69fd9ba3214c9376
previewMode: "1000"
config:
  condition: customer_id IS NOT NULL AND order_id IS NOT NULL
input:
  - node: filter_3
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "customer_id IS NOT NULL AND order_id IS NOT NULL"
}
inputs = {
    "data": ctx["filter_3.filtered_data"]
}
out = run(config, inputs, spark)
ctx["filter_5.filtered_data"] = out["filtered_data"]
ctx["filter_5.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["filter_5.filtered_data"])
    display(ctx["filter_5.excluded_data"])

In [0]:
"""
id: prepare_6
template: prepare
templateVersion: 1.0.0
name: Data cleanup and drop columns
position:
  x: 2068
  y: 520
description:
  text: Set negative total amounts to zero, fill missing quantities with zero, and set ingested time, source file, and recovered data fields to null.
  hash: 22aa1c93
previewCodeHash: 122540685d6374a8
previewMode: "1000"
config:
  actions:
    - type: formula
      target: total_amount
      expression: IF(CAST(total_amount AS DOUBLE) < 0, 0, CAST(total_amount AS DOUBLE))
    - type: fill_null
      column: quantity
      with: "0"
input:
  - node: filter_5
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "total_amount",
            "expression": "IF(CAST(total_amount AS DOUBLE) < 0, 0, CAST(total_amount AS DOUBLE))"
        },
        {
            "type": "fill_null",
            "column": "quantity",
            "with": "0"
        }
    ]
}
inputs = {
    "data": ctx["filter_5.filtered_data"]
}
out = run(config, inputs, spark)
ctx["prepare_6.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["prepare_6.prepared_data"])

In [0]:
"""
id: select_6
template: transform
templateVersion: 3.0.0
name: Final_table
position:
  x: 2328
  y: 520
description:
  text: Exclude _rescued_data, source_file, and ingested_at columns.
  hash: "75611721"
previewCodeHash: b0144f5b8041a7c3
previewMode: "1000"
config:
  mode: passthrough
  edits:
    - column: _rescued_data
      checked: false
    - column: source_file
      checked: false
    - column: ingested_at
      checked: false
  ordered: []
input:
  - node: prepare_6
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
from pyspark.sql import functions as F
from pyspark.sql import types as _spark_types
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DateType,
    DecimalType,
    FractionalType,
    IntegerType,
    IntegralType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_TIMESTAMP_TYPES = tuple(
    t
    for t in (TimestampType, getattr(_spark_types, "TimestampNTZType", None))
    if t is not None
)

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def _type_category(data_type) -> Optional[str]:
    if isinstance(data_type, BooleanType):
        return "boolean"
    if isinstance(data_type, DecimalType):
        return "decimal"
    if isinstance(data_type, IntegralType):
        return "integer"
    if isinstance(data_type, FractionalType):
        return "float"
    if isinstance(data_type, StringType):
        return "string"
    if isinstance(data_type, DateType):
        return "date"
    if isinstance(data_type, _TIMESTAMP_TYPES):
        return "timestamp"
    if isinstance(data_type, BinaryType):
        return "binary"
    return None

_METADATA_SCHEMA = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Type", StringType(), False),
        StructField("Category", StringType(), True),
        StructField("FieldNumber", IntegerType(), False),
        StructField("IsNumeric", BooleanType(), False),
        StructField("IsInteger", BooleanType(), False),
        StructField("IsFloat", BooleanType(), False),
        StructField("IsString", BooleanType(), False),
        StructField("IsDateOrTime", BooleanType(), False),
        StructField("IsBinary", BooleanType(), False),
    ]
)

def _metadata_row(index: int, field):
    data_type = field.dataType
    category = _type_category(data_type)
    return (
        field.name,
        data_type.simpleString(),
        category,
        index + 1,
        category in ("integer", "float", "decimal"),
        isinstance(data_type, IntegralType),
        category == "float",
        isinstance(data_type, StringType),
        isinstance(data_type, (DateType,) + _TIMESTAMP_TYPES),
        isinstance(data_type, BinaryType),
    )

def _sql_string_literal(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

def _predicate_for(dynamic: Dict[str, Any]) -> str:
    kind = dynamic.get("kind")
    if kind == "byType":
        selected = dynamic.get("types") or []
        if not selected:
            return "false"
        quoted = ", ".join(_sql_string_literal(t) for t in selected)
        return "Category IN (" + quoted + ")"
    if kind == "byName":
        pattern = dynamic.get("namePattern")
        if not pattern or not pattern.get("op"):
            raise ValueError(
                "Select: a byName dynamic rule requires namePattern with an 'op' and 'value'"
            )
        op = pattern.get("op")
        value = pattern.get("value", "")
        case_sensitive = pattern.get("caseSensitive", False) is True
        if op == "regex":
            regex = value if case_sensitive else "(?i)" + value
            return "Name rlike " + _sql_string_literal(regex)
        name_expr = "Name" if case_sensitive else "lower(Name)"
        needle = value if case_sensitive else value.lower()
        escaped = needle.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        if op == "startsWith":
            like = escaped + "%"
        elif op == "endsWith":
            like = "%" + escaped
        else:
            like = "%" + escaped + "%"
        return name_expr + " like " + _sql_string_literal(like)
    if kind == "byExpression":
        expression = dynamic.get("expression") or ""
        return expression if expression.strip() else "false"
    return "false"

def _resolve_dynamic(df, spark, dynamic: Dict[str, Any]):
    predicate = _predicate_for(dynamic)
    fields = df.schema.fields
    rows = [_metadata_row(i, f) for i, f in enumerate(fields)]
    meta_df = spark.createDataFrame(rows, _METADATA_SCHEMA)
    matched_ordinals = {
        row["FieldNumber"] for row in meta_df.filter(predicate).select("FieldNumber").collect()
    }
    action = dynamic.get("action", "keep")
    return [
        index
        for index in range(len(fields))
        if ((index + 1) in matched_ordinals) != (action == "remove")
    ]

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    dynamic = config.get("dynamic")
    if dynamic:
        kept_indexes = _resolve_dynamic(df, spark, dynamic)
        original_names = [field.name for field in df.schema.fields]
        placeholders = ["_c" + str(i) for i in range(len(original_names))]
        projected = df.toDF(*placeholders).select(*(F.col(placeholders[i]) for i in kept_indexes))
        return {
            "transformed_data": projected.toDF(*(original_names[i] for i in kept_indexes))
        }

    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "passthrough",
    "edits": [
        {
            "column": "_rescued_data",
            "checked": False
        },
        {
            "column": "source_file",
            "checked": False
        },
        {
            "column": "ingested_at",
            "checked": False
        }
    ],
    "ordered": []
}
inputs = {
    "data": ctx["prepare_6.prepared_data"]
}
out = run(config, inputs, spark)
ctx["select_6.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["select_6.transformed_data"])